In [1]:
%%capture
# Load SQL extension and connect to DuckDB
%load_ext sql
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 50
%sql duckdb://

### **Monthly Revenue Performance**

How has monthly revenue changed over time, and how many orders and customers are driving that revenue?

In [2]:
%%sql

WITH monthly_revenue AS (
SELECT 
    strftime(order_date, '%Y-%m') AS month,
    COUNT(order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(SUM(revenue), 0)::INT AS total_revenue
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY month
ORDER BY month
)

SELECT
    *
FROM monthly_revenue
WHERE month = (SELECT MIN(month) FROM monthly_revenue) OR month = (SELECT MAX(month) FROM monthly_revenue)
;

month,total_orders,unique_customers,total_revenue
2024-01,42,41,32088
2026-08,1939,1166,1466817


In the past 2.5 years, Northstar has grown from 41 customers and `$32,088` in revenue to 1,166 customers and `$1,466,817` in revenue. 


********


### Revenue by Acquisition Channel

Which customer acquisiton channels generate the most in total revenue?

In [3]:
%%sql

SELECT 
    RANK() OVER (ORDER BY SUM(o.revenue) DESC) AS rank,
    mc.channel_name AS channel_name,
    ROUND(SUM(o.revenue), 0)::INT AS total_revenue
FROM 'data/orders.csv' o
LEFT JOIN 'data/customers.csv' c
    ON o.customer_id = c.customer_id
LEFT JOIN 'data/marketing_channels.csv' mc
    ON c.channel_id = mc.channel_id
WHERE o.order_status = 'Completed'
GROUP BY channel_name
ORDER BY total_revenue DESC
;

rank,channel_name,total_revenue
1,Organic,6764476
2,Google Ads,4503094
3,Referral,4450311
4,Outbound Sales,3622330
5,Partner,1313463
6,Social,957119
7,Content,647671
8,Events,458620


Organic channels generate the most in revenue whereas Events generates the least.


--------


### Top 25 Customers by lifetime order revenue


In [4]:
%%sql

SELECT
    RANK() OVER(ORDER BY SUM(revenue) DESC) AS order_revenue_rank,
    customer_id,
    ROUND(SUM(revenue), 0)::INT AS total_order_revenue
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY customer_id
LIMIT 25
;

order_revenue_rank,customer_id,total_order_revenue
1,11909,15970
2,14256,15079
3,10398,14604
4,19314,14113
5,13897,13421
6,19590,13020
7,17100,12905
8,11870,12708
9,13741,12437
10,19238,12158


Top 25 customers have ordered over `$10,000` in lifetime revenue where the top spender has almost hit `$16,000+`.

--------------

### Repeat Customer Rate

What percentage of customers have placed more than one order? How does that percentage differ by company size?

In [5]:
%%sql

WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS order_count
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
)
SELECT
    ROUND(100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS repeat_customer_pct
FROM customer_orders
;

repeat_customer_pct
85.17


In [6]:
%%sql

WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS order_count
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
),
customer_sizes AS (
    SELECT
        c.company_size,
        CASE WHEN co.order_count > 1 THEN 1 ELSE 0 END AS is_repeat
    FROM 'data/customers.csv' c
    LEFT JOIN customer_orders co
        ON c.customer_id = co.customer_id
)
SELECT
    company_size,
    ROUND(100.0 * SUM(is_repeat)/COUNT(*), 2) AS repeat_customer_pct
FROM customer_sizes
GROUP BY company_size
ORDER BY repeat_customer_pct DESC
;

company_size,repeat_customer_pct
Mid-Market,82.16
Enterprise,81.7
SMB,80.59


------

### Month-over-Month Average Order Value (AOV)

How has the company's average order value changed month over month, and what is the percentage change compared to the previous month?

In [7]:
%%sql

WITH monthly_avg_revenue AS (
    SELECT 
        strftime('%Y-%m', order_date) AS month,
        AVG(revenue) AS avg_revenue,
        LAG(avg_revenue) OVER(ORDER BY month) AS previous_month_avg_revenue
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY month
)
SELECT 
    month,
    ROUND(avg_revenue, 2) AS avg_revenue,
    ROUND(previous_month_avg_revenue, 2) AS previous_month_avg_revenue,
    ROUND(avg_revenue - previous_month_avg_revenue, 2) AS avg_revenue_change,
    ROUND(100.0 * (avg_revenue - previous_month_avg_revenue) / previous_month_avg_revenue, 2) AS avg_revenue_change_pct
FROM monthly_avg_revenue
;

month,avg_revenue,previous_month_avg_revenue,avg_revenue_change,avg_revenue_change_pct
2024-01,764.01,None,None,None
2024-02,742.64,764.01,-21.36,-2.8
2024-03,663.94,742.64,-78.7,-10.6
2024-04,768.91,663.94,104.96,15.81
2024-05,754.32,768.91,-14.58,-1.9
2024-06,751.4,754.32,-2.93,-0.39
2024-07,698.46,751.4,-52.94,-7.05
2024-08,720.85,698.46,22.39,3.21
2024-09,682.15,720.85,-38.7,-5.37
2024-10,707.71,682.15,25.56,3.75


-----

### Cohort Retention Analysis

In [8]:
%%sql

WITH cohort AS (
    SELECT 
        customer_id,
        strftime('%Y-%m', MIN(order_date)) AS cohort_month
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
),
activity AS (
    SELECT DISTINCT
        customer_id,
        strftime('%Y-%m', order_date) AS activity_month
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
),
cohort_activity AS (
    SELECT
        c.customer_id,
        c.cohort_month,
        a.activity_month,
        date_diff('month', strptime(c.cohort_month, '%Y-%m'), strptime(a.activity_month, '%Y-%m')) AS months_since_cohort
    FROM cohort c
    LEFT JOIN activity a ON c.customer_id = a.customer_id
),
cohort_retention AS (
    SELECT
        cohort_month,
        months_since_cohort,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM cohort_activity
    GROUP BY cohort_month, months_since_cohort
),
cohort_groups AS (
    SELECT 
        cohort_month,
        active_customers
    FROM cohort_retention
    WHERE months_since_cohort = 0
    ORDER BY cohort_month
)
SELECT
    cr.cohort_month,
    cr.months_since_cohort,
    cr.active_customers,
    cg.active_customers AS cohort_size,
    ROUND(100.0*cr.active_customers/cg.active_customers, 2) AS retention_rate
FROM cohort_retention cr
LEFT JOIN cohort_groups cg 
    ON cr.cohort_month = cg.cohort_month
ORDER BY cr.cohort_month, cr.months_since_cohort
;

cohort_month,months_since_cohort,active_customers,cohort_size,retention_rate
2024-01,0,41,41,100.0
2024-01,1,6,41,14.63
2024-01,2,3,41,7.32
2024-01,3,4,41,9.76
2024-01,4,4,41,9.76
2024-01,5,6,41,14.63
2024-01,6,1,41,2.44
2024-01,7,5,41,12.2
2024-01,8,1,41,2.44
2024-01,9,2,41,4.88


-----

### Customer Lifetime Value

What does customer lifetime value look like across the customer base, and what factors are associated with higher-value customers?

In [9]:
%%sql

SELECT
    customer_id,
    ROUND(SUM(revenue), 2) AS lifetime_revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS most_recent_order_date,
    date_diff('month', first_order_date, most_recent_order_date) AS months_as_customer,
    ROUND(lifetime_revenue / total_orders, 2) AS avg_order_value
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY customer_id
ORDER BY lifetime_revenue DESC
LIMIT 10
;

customer_id,lifetime_revenue,total_orders,first_order_date,most_recent_order_date,months_as_customer,avg_order_value
11909,15970.3,7,2025-05-26,2026-08-25,15,2281.47
14256,15079.35,7,2025-07-22,2026-01-10,6,2154.19
10398,14604.41,7,2024-03-05,2026-07-29,28,2086.34
19314,14112.66,5,2024-12-01,2026-07-12,19,2822.53
13897,13421.3,7,2024-09-20,2026-04-30,19,1917.33
19590,13020.21,14,2024-03-25,2026-07-12,28,930.02
17100,12905.38,10,2024-08-07,2026-06-08,22,1290.54
11870,12708.05,8,2026-06-18,2026-06-18,0,1588.51
13741,12437.03,4,2025-06-13,2026-08-07,14,3109.26
19238,12158.2,9,2024-07-05,2026-08-19,25,1350.91


First, we create a CTE of completed customer orders and lifetime value metrics with how many orders have been processed since their first purchase date.


In [10]:

%%sql

WITH customer_lifetime_revenue AS ( 
    SELECT
        customer_id,
        SUM(revenue) AS lifetime_revenue,
        COUNT(DISTINCT order_id) AS total_orders,
        MIN(order_date) AS first_order_date,
        MAX(order_date) AS most_recent_order_date,
        date_diff('month', first_order_date, most_recent_order_date) AS months_as_customer,
        lifetime_revenue / total_orders AS avg_order_value
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
)
SELECT
    ROUND(AVG(lifetime_revenue), 2) AS avg_lifetime_revenue,
    ROUND(MEDIAN(lifetime_revenue), 2) AS median_lifetime_revenue
FROM customer_lifetime_revenue

avg_lifetime_revenue,median_lifetime_revenue
2381.5,1893.85


Average and Median lifetime revenue are closely aligned so that tells us that not a lot of high-value customers are pulling our average upward.

In [11]:
%%sql

WITH customer_lifetime_revenue AS ( 
    SELECT
        customer_id,
        SUM(revenue) AS lifetime_revenue,
        COUNT(DISTINCT order_id) AS total_orders,
        MIN(order_date) AS first_order_date,
        MAX(order_date) AS most_recent_order_date,
        date_diff('month', first_order_date, most_recent_order_date) AS months_as_customer,
        lifetime_revenue / total_orders AS avg_order_value
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
)
SELECT
    c.company_size AS company_size, 
    ROUND(AVG(r.lifetime_revenue), 2) AS avg_lifetime_revenue,
    ROUND(MEDIAN(r.lifetime_revenue), 2) AS median_lifetime_revenue,
    ROUND(AVG(r.avg_order_value), 2) AS avg_customer_aov,
    ROUND(AVG(r.total_orders), 2) AS avg_total_orders,
    COUNT(r.customer_id) AS customer_count
FROM customer_lifetime_revenue r
LEFT JOIN 'data/customers.csv' c
    ON r.customer_id = c.customer_id
GROUP BY c.company_size

company_size,avg_lifetime_revenue,median_lifetime_revenue,avg_customer_aov,avg_total_orders,customer_count
Enterprise,2414.12,1813.58,725.15,3.27,1401
Mid-Market,2362.33,1889.82,718.99,3.24,2974
SMB,2383.68,1909.32,732.87,3.24,5164


Customer company size showed limited differentiation in terms of realized lifetime revenue. Enterprise customers had the highest average lifetime revenue, but the SMB customers had a higher median lifetime revenue, suggesting that some high-value Enterprise customers skewed the average upwards. The average order value and total orders were also similar across company sizes, indicating that customer behavior was relatively consistent regardless of company size.

----

### Revenue Concentration

What percentage of total revenue comes from the top 10% and 20% of customers?

In [12]:
%%sql

WITH revenue_decile AS (
SELECT
    customer_id,
    SUM(revenue) AS lifetime_revenue,
    NTILE(10) OVER (ORDER BY SUM(revenue) DESC) AS revenue_decile
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY customer_id
)

SELECT
    ROUND(SUM(CASE WHEN revenue_decile = 1 THEN lifetime_revenue ELSE 0 END), 2) AS top_10_pct_revenue,
    ROUND(SUM(lifetime_revenue), 2) AS total_revenue,
    ROUND(100.0*top_10_pct_revenue/total_revenue, 2) AS top_10_revenue_pct
FROM revenue_decile

top_10_pct_revenue,total_revenue,top_10_revenue_pct
6324021.45,22717084.74,27.84


In [13]:
%%sql

WITH revenue_decile AS (
SELECT
    customer_id,
    SUM(revenue) AS lifetime_revenue,
    NTILE(5) OVER (ORDER BY SUM(revenue) DESC) AS revenue_decile
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY customer_id
)

SELECT
    ROUND(SUM(CASE WHEN revenue_decile = 1 THEN lifetime_revenue ELSE 0 END), 2) AS top_20_pct_revenue,
    ROUND(SUM(lifetime_revenue), 2) AS total_revenue,
    ROUND(100.0*top_20_pct_revenue/total_revenue, 2) AS top_20_revenue_pct
FROM revenue_decile

top_20_pct_revenue,total_revenue,top_20_revenue_pct
10436410.52,22717084.74,45.94


<div align="left">

| Top n customers | Revenue % |
| :--- | :--- |
| 10% | 27.84% |
| 20% | 45.94% |

</div>

This shows us that there is a meaningful concentration of revenue from high-value customers.

----

### Detecting Subscription Upgrades/Downgrades

Which customers upgraded or downgraded their subscription, and when did those changes occur?

In [14]:
%%sql

WITH product_info AS (
    SELECT
    *,
    CASE WHEN product_name ILIKE '%starter%' THEN 'Starter'
    WHEN product_name ILIKE '%pro%' THEN 'Pro'
    WHEN product_name ILIKE '%enterprise%' THEN 'Enterprise'
    END AS tier
FROM 'data/products.csv'
),
subscription_w_tiers AS (
    SELECT
        s.*,
        p.tier,
        CASE WHEN tier = 'Starter' THEN 1
        WHEN tier = 'Pro' THEN 2
        WHEN tier = 'Enterprise' THEN 3
        END AS tier_num
    FROM 'data/subscriptions.csv' s
    LEFT JOIN product_info p
        ON s.product_id = p.product_id
),
subscription_w_history AS (
    SELECT
        customer_id,
        product_id,
        start_date,
        tier,
        LAG(tier) OVER(PARTITION BY customer_id ORDER BY start_date) AS prev_tier,
        tier_num,
        LAG(tier_num) OVER(PARTITION BY customer_id ORDER BY start_date) AS prev_tier_num
    FROM subscription_w_tiers
),
subscription_change AS (
    SELECT
        customer_id,
        product_id,
        start_date,
        tier,
        prev_tier,
        CASE WHEN tier_num > prev_tier_num THEN 'Upgrade'
        WHEN tier_num < prev_tier_num THEN 'Downgrade'
        ELSE 'No Change'
        END AS subscription_change
    FROM subscription_w_history

)

SELECT
    subscription_change,
    COUNT(*) AS num_subscriptions
FROM subscription_change
GROUP BY subscription_change
;

subscription_change,num_subscriptions
Upgrade,2951
Downgrade,2932
No Change,11117


Around ~11,000 subscriptions, customers have kept the same subscription tier or swapped to a different subscription type with the same tier.

In [15]:
%%sql

WITH product_info AS (
    SELECT
    *,
    CASE WHEN product_name ILIKE '%starter%' THEN 'Starter'
    WHEN product_name ILIKE '%pro%' THEN 'Pro'
    WHEN product_name ILIKE '%enterprise%' THEN 'Enterprise'
    END AS tier
FROM 'data/products.csv'
),
subscription_w_tiers AS (
    SELECT
        s.*,
        p.tier,
        CASE WHEN tier = 'Starter' THEN 1
        WHEN tier = 'Pro' THEN 2
        WHEN tier = 'Enterprise' THEN 3
        END AS tier_num
    FROM 'data/subscriptions.csv' s
    LEFT JOIN product_info p
        ON s.product_id = p.product_id
),
subscription_w_history AS (
    SELECT
        customer_id,
        product_id,
        start_date,
        tier,
        LAG(tier) OVER(PARTITION BY customer_id ORDER BY start_date) AS prev_tier,
        tier_num,
        LAG(tier_num) OVER(PARTITION BY customer_id ORDER BY start_date) AS prev_tier_num
    FROM subscription_w_tiers
),
subscription_change AS (
    SELECT
        customer_id,
        product_id,
        start_date,
        tier,
        prev_tier,
        CASE WHEN tier_num > prev_tier_num THEN 'Upgrade'
        WHEN tier_num < prev_tier_num THEN 'Downgrade'
        ELSE 'No Change'
        END AS subscription_change
    FROM subscription_w_history

)

SELECT
    start_date.strftime('%Y-%m') AS month,
    SUM(CASE WHEN subscription_change = 'Upgrade' THEN 1 ELSE 0 END) AS Upgrades,
    SUM(CASE WHEN subscription_change = 'Downgrade' THEN 1 ELSE 0 END) AS Downgrades
FROM subscription_change
WHERE subscription_change IN ('Upgrade', 'Downgrade')
GROUP BY month
ORDER BY month
LIMIT 12
;

month,Upgrades,Downgrades
2024-01,3,2
2024-02,8,6
2024-03,21,17
2024-04,35,39
2024-05,53,56
2024-06,61,74
2024-07,81,85
2024-08,91,103
2024-09,93,106
2024-10,115,83


----

### Failed Payments = Churn?

Are failed payments associated with customers subsequently churning?

In [37]:
%%sql

WITH cancellations AS (
SELECT 
    s1.*,
    ROW_NUMBER() OVER(PARTITION BY s1.customer_id ORDER BY s1.end_date DESC) AS cancellation_rank
FROM 'data/subscriptions.csv' s1
LEFT JOIN 'data/subscriptions.csv' s2
    ON s1.customer_id = s2.customer_id
    AND s2.start_date > s1.end_date
WHERE s1.status = 'Cancelled' AND s2.subscription_id IS NULL
ORDER BY s1.customer_id, s1.start_date
)

SELECT
    customer_id,
    end_date AS churn_
FROM cancellations
WHERE cancellation_rank = 1
LIMIT 10
;

customer_id,churn_date
10002,2026-08-31
10004,2026-08-09
10012,2026-07-09
10018,2025-09-22
10020,2025-11-02
10022,2026-02-11
10024,2025-11-02
10028,2026-01-29
10030,2024-12-30
10033,2026-08-31


The above data shows us final churn date. Customers could cancel one subscription and swap to another subscription but this would be considered subscription change rather than churn. We find final churn here by completing a self join and filtering to the customers who have fully churned. Also, customers could have multiple cancellations so we want to look at the end_date that is most recent to find final churn date.

In [64]:
%%sql

WITH failed_payments AS (
    SELECT
        customer_id,
        payment_date,
        payment_status
    FROM 'data/payments.csv'
    WHERE payment_status = 'Failed'
),
cancellations AS (
    SELECT 
        s1.*,
        ROW_NUMBER() OVER(PARTITION BY s1.customer_id ORDER BY s1.end_date DESC) AS cancellation_rank
    FROM 'data/subscriptions.csv' s1
    LEFT JOIN 'data/subscriptions.csv' s2
        ON s1.customer_id = s2.customer_id
        AND s2.start_date > s1.end_date
    WHERE s1.status = 'Cancelled' AND s2.subscription_id IS NULL
    ORDER BY s1.customer_id, s1.start_date
),
customer_payment_status AS (
    SELECT
        c.customer_id,
        CASE WHEN f.payment_date IS NOT NULL THEN 1
        ELSE 0
        END AS had_failed_payment,
        c.customer_status
    FROM 'data/customers.csv' c
    LEFT JOIN failed_payments f
        ON c.customer_id = f.customer_id
)

SELECT
    had_failed_payment,
    COUNT(DISTINCT CASE WHEN customer_status = 'Churned' THEN customer_id END) AS churned_customers,
    COUNT(DISTINCT customer_id) AS total_customers,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN customer_status = 'Churned' THEN customer_id END) / COUNT(DISTINCT customer_id), 2) AS churn_pct,

FROM customer_payment_status
GROUP BY had_failed_payment

;


had_failed_payment,churned_customers,total_customers,churn_pct
0,1145,8210,13.95
1,254,1790,14.19


Failed payment customers have a 0.24 percentage point higher observed churn rate.
